In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, to_timestamp

spark = SparkSession.builder \
    .appName("Bronzelayer") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,"
        "io.delta:delta-spark_2.12:3.1.0"
    ) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

26/04/29 13:31:05 WARN Utils: Your hostname, Saileshs-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.3 instead (on interface en0)
26/04/29 13:31:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/saileshpola/.ivy2/cache
The jars for the packages stored in: /Users/saileshpola/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-26c9caa0-e55e-4bf4-8808-a75d519259ba;1.0
	confs: [default]


:: loading settings :: url = jar:file:/Users/saileshpola/PycharmProjects/PythonProject/PysparkKafkaETE/.venv/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.0 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 232ms :: artifacts dl 5ms
	:: modules in use:
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	commons-logging#commons-logging;1.1.3 from central in [default]
	io.delta#

In [2]:
df = spark.readStream\
    .format("kafka")\
    .option("kafka.bootstrap.servers", "localhost:9092")\
    .option("subscribe", "trades")\
    .option("startingOffsets", "latest")\
    .option("maxOffsetsPerTrigger", 100)\
    .load()

In [3]:
df1 = df.selectExpr("CAST(value AS STRING) as value",
                    "topic",
                    "partition",
                    "offset",
                    "timestamp as kafka_timestamp")\
    .withColumn("ingestion_time",current_timestamp())


In [4]:
from pyspark.sql.types import *
from pyspark.sql.functions import from_json, col, to_json, date_format

trade_schema = StructType([
    StructField("trade_id", StringType(), True),
    StructField("trader_id", StringType(), True),
    StructField("symbol", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("trade_timestamp", StringType(), True),
    StructField("ingestion_timestamp", StringType(), True),
    StructField("exchange", StringType(), True),
    StructField("side", StringType(), True),
    StructField("metadata", StructType([
        StructField("source", StringType(), True),
        StructField("version", IntegerType(), True)
    ]), True)
])

In [5]:
df_parsed = df1.withColumn('data', from_json('value', trade_schema))

In [6]:
df_final = df_parsed.select('data.*',
                            "topic",
                            "partition",
                            "offset",
                            "kafka_timestamp",
                            "ingestion_time",
                            )

In [7]:
query = (df_final.writeStream.format("delta")
        .outputMode("append")
        .option("checkpointLocation", 'checkpoints/bronze/trades')
        .option("path", "data/bronze/trades")
        .start())